# news_sentiment_pipeline

In [ ]:
import numpy as np
import pandas as pd
import os, gc
from tqdm.auto import tqdm
from datetime import datetime, timezone, timedelta

In [ ]:
df = pd.read_excel('../data/탭별_20년도_1_3월.xlsx', header=None)
df.columns = ['Nan', 'date', 'title', 'description', 'url']
df = df.drop(df.columns[0], axis=1)
df = df.drop(df.index[0], axis=0)
df.head()

In [ ]:
len(df)

In [ ]:
import re
df['description'] = df['description'].apply(lambda x: re.sub(r"\n+", " ", x).strip())

In [ ]:
from transformers import pipeline, set_seed

In [ ]:
import torch
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration

### @title kobart (ainize)

In [ ]:
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration
tokenizer = PreTrainedTokenizerFast.from_pretrained("ainize/kobart-news")
model = BartForConditionalGeneration.from_pretrained("ainize/kobart-news")

In [ ]:
df['description'][889]

In [ ]:
df['description'][398]

In [ ]:
import random

def generate_summary(news_text):

    deterministic = True

    random.seed(123)
    np.random.seed(123)
    torch.manual_seed(123)
    torch.cuda.manual_seed_all(123)
    if deterministic:
       torch.backends.cudnn.deterministic = True
       torch.backends.cudnn.benchmark = False

    inputs = tokenizer.encode(news_text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(inputs, max_length=80, min_length=40, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

df['summary'] = df['description'].apply(generate_summary)

for idx, summary in enumerate(df['summary']):
    print(f"Summary {idx + 1}: {summary}")

In [ ]:
print(df['summary'])

In [ ]:
# 쓸데없는 내용이 들어있는 인덱스는 삭제

remove = [1, 2, 3, 105, 114, 119, 120, 194, 211, 284, 308, 391, 397, 398, 553, 554, 555, 677, 678, 721, 733, 736, 755, 820, 847, 883, 889, 962, 992, \
          1132, 1134, 1145, 1278, 1279, 1470, 1577, 1738, 1740, 1741, 1743, 1744, 1746]

df.drop(remove, inplace=True)
df.reset_index(drop=True, inplace=True)

print(df)

In [ ]:
## 전처리

import re

def remove_special_characters(text):
    cleaned_text = re.sub(r'[■●▶▲◆△#@\\]', '', text)
    cleaned_text = re.sub(r'\[.*?\]', '', cleaned_text)
    cleaned_text = re.sub(r'\(.*?\)', '', cleaned_text)
    return cleaned_text

df['summary'] = df['summary'].apply(remove_special_characters)

In [ ]:
def remove_newlines(text):
    return text.replace('\n', '')

df['summary'] = df['summary'].apply(remove_newlines)

In [ ]:
df['summary']

In [ ]:
df.to_excel("요약_20년_1_3월.xlsx", sheet_name='Sheet1', index=False)

In [ ]:
!pip install openai

In [ ]:
import openai
import os

openai.api_key = 'YOUR_KEY'

In [ ]:
def create_sentiment_prompt(text):

    return f"Please analyze the sentiment of the following text and classify it as positive, negative, or neutral: {text}"

def analyze_sentiment(text):

    prompt = create_sentiment_prompt(text)

    response = openai.Completion.create(

        engine="text-davinci-002",

        prompt=prompt,

        max_tokens=50,

        n=1,

        stop=None,

        temperature=0.5,

    )

    sentiment = response.choices[0].text.strip()

    return sentiment

df['sentiment'] = df['summary'].apply(analyze_sentiment)

In [ ]:
df['sentiment']

In [ ]:
def assign_value(sentiment):
    if 'positive' in sentiment:
        return 1
    elif 'negative' in sentiment:
        return -1
    else:
        return 0

df['value'] = df['sentiment'].apply(assign_value)

In [ ]:
df.head()

In [ ]:
df.to_excel("감성분석_22년도_4_6월.xlsx", sheet_name='Sheet1', index=False)

In [ ]:
#!pip install nltk

In [ ]:
## BLEU 성능

import nltk.translate.bleu_score as bleu

candidate = """집값 전망 "상승" 35% "유지" 29% "하락" 28%고연령층·새 정부 지지층일수록 집값 하락 전망 지난달 29일 서울의 한 부동산업체 모습. 최근 수도권 부동산 지표는 규제 완화 기대 심리로 반등했다. 뉴시스여론조사업체 한국갤럽이 1일 공개한 여론조사 결과, 집값이 상승할 것이라는 전망이 2020년 6월 조사 이후 2년여 만에 30%대로 하락했다. 새 정부의 부동산 정책에 대한 기대감이 반영된 것인데, 정치 성향과 연령에 따라 입장은 엇갈렸다.한국갤럽이 지난달 29∼31일 성인 유권자 1,001명을 대상으로 조사한 결과(표본오차 95% 신뢰수준에서 ±3.1%포인트)에 따르면, 향후 1년 집값 전망을 묻는 질문에 35%가 \'오른다\'고 답했다. \'변화 없을 것\'이라는 응답은 29%, \'내린다\'는 응답은 28%였다.집값 상승 전망 순지수(상승 전망 비율에서 하락 전망 비율을 뺀 수치)는 직전 조사였던 2021년 9월 조사(43)에서 7로 떨어졌다.\'오른다\'는 응답의 비중이 30%대로 낮아진 것은 2020년 6월 조사(37%) 이후 약 1년 10개월 만이다. 집값 상승 전망은 2020년 7월 조사에서 61%로 치솟은 후 지속적으로 높게 유지돼 왔다. 여론조사 결과는 새 정부 출범 후 집값이 안정화 또는 하향할 것으로 기대하는 여론이 반영된 것으로 보인다. 특히 현 정부를 지지할수록, 고령층일수록 집값이 하락할 것으로 기대하는 태도가 나타난다.응답자 특성별로 집값 상승 전망 순지수를 보면 지역은 대구·경북(-11), 연령은 60대(-4)와 70대 이상(-9), 윤석열 대통령 당선인 국정수행 전망 긍정평가 응답자(-5)가 집값 하락을 전망하는 비중이 우세했다.반면 학생(79)과 윤석열 당선인 국정수행 전망 부정평가 응답자(21), 18∼29세(19) 30대(13) 40대(15) 등은 집값 상승을 전망하는 비중이 높은 것으로 나타났다."본인 소유 집 있어야" 2014년 54% → 2019년 79% 지난달 31일 서울 송파구 롯데월드타워에서 바라본 송파구 일대의 아파트 모습. 고영권 기자본인 소유의 집이 있어야 하는지를 묻는 질문에는 "있어야 한다"는 응답의 비중이 79%로 나타났다. "그럴 필요 없다"는 응답은 19%에 그쳤다. 2014년 7월 조사에서는 \'내 집이 있어야 한다\'가 54%였으나 2017년 1월 63%, 2019년 3월 72%, 2022년 3월 79%로 늘었다.무주택자들은 대체로 이른 시일 안에 본인 소유의 집을 사지는 못할 것으로 내다봤다. 본인 소유 주택 구매 예상 시기를 5년 미만으로 본 비중은 7%, 5∼10년 내는 34%, 10년 이상은 23%였다. \'영영 구매하기 어려울 것\'이라는 응답은 18%였고, \'내 집 마련 의향이 없다\'는 응답은 9%였다.연령별로는 30대(55%)와 40대(42%) 대부분이 5∼10년 내에 집을 구매할 것으로 내다본 반면 20대는 10년 이상(40%)이 소요될 것으로 내다봤다. 60대와 70대 이상은 \'영영 어려울 것 같다\'는 응답의 비중이 각각 48%, 46%로 가장 높았다.경기와 살림살이에 대한 전망에는 개인의 정치 성향이 크게 영향을 미쳤다. 일반적으로 경기 낙관론은 정부 정책 방향에 공감하는 쪽에서 강하고 비관론은 정부에 비판적인 쪽에서 강하다. 정권 교체기가 도래하면서 국민의힘 지지층, 보수 성향 응답자들이 대거 낙관론으로 돌아선 반면 더불어민주당 지지층과 진보 성향 응답자들은 비관적인 태도로 바뀌었다."""
references = [
    "변화 없을 것'이라는 응답은 29%, '내린다'는 응답은 28%였다.집값 상승 전망 순지수(상승 전망 비율에서 하락 전망 비율을 뺀 수치)는 직전 조사였던 2021년 9월 조사(43)에서 7로 떨어졌다.'오른다'는 응답의 비중이 30%대로 낮아진 것은 2020년 6월 조사(37%) 이후 약 1년 10개월 만이다."]

print('BLEU:', bleu.sentence_bleu([list(map(lambda ref: ref.split(), references))], candidate.split()))

# feature_dataset_merge

In [ ]:
df1 = pd.read_excel('../data/아파트 매매가격지수.xlsx')

In [ ]:
import requests

api_key =  "YOUR_API_KEY"
url = 'https://ecos.bok.or.kr/api/StatisticSearch/{api_key}/json/kr/1/100/722Y001/D/20210101/20230331'
response = requests.get(url)
result = response.json()
list_total_count=(int)(result['StatisticSearch']['list_total_count'])
list_count=(int)(list_total_count/100) + 1


rows=[]
for i in range(0,list_count):
    start = str(i * 100 + 1)
    end = str((i + 1) * 100)

    url = 'https://ecos.bok.or.kr/api/StatisticSearch/21SC44F70ZNCSG2AMIWU/json/kr/' \
            + start + '/' + end + '/722Y001/D/20210101/20230331'
    response = requests.get(url)
    result = response.json()
    rows = rows + result['StatisticSearch']['row']

df2 =pd.DataFrame(rows)
df2


In [ ]:
df1=df2[['ITEM_NAME1','TIME','DATA_VALUE']]
df1['date']=pd.to_datetime((df1['TIME'].str[:4] + '-' + df1['TIME'].str[4:6] + '-' + df1['TIME'].str[6:8]))
df1=df1.astype({'DATA_VALUE':'float'})
df1=df1.drop_duplicates()
df1

In [ ]:
import yfinance as yf
from datetime import datetime

enddate=datetime.now().strftime('%Y-%m-%d')
kospi=yf.download('^KS11', '2021-01-01', '2023-03-31', auto_adjust=True)
kospi

# time_series_preprocessing

In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rc('font', family='NanumBarunGothic')

### @title 매매가격지수 (예측값)

In [ ]:
df = pd.read_excel('../data/아파트 매매가격지수.xlsx')
df.head()

In [ ]:
df.tail(10)

In [ ]:
# 전국의 매매가격지수와 해당하는 기간만 지정해 전처리
price = df.iloc[list(range(422, 449)), [0, 1]]
price

In [ ]:
price.loc[422, '구분'] = '2021-1'
price.head()

In [ ]:
price['구분'] = ['2021-1', '2021-2', '2021-3', '2021-4', '2021-5', '2021-6', '2021-7', '2021-8', '2021-9', '2021-10', '2021-11', '2021-12',
                 '2022-1', '2022-2', '2022-3', '2022-4', '2022-5', '2022-6', '2022-7', '2022-8', '2022-9', '2022-10', '2022-11', '2022-12',
                 '2023-1', '2023-2', '2023-3']
price

In [ ]:
price.columns = ['시점', '매매가격지수']
price['시점'] = pd.to_datetime(price['시점'], format='%Y-%m')
price.head()

In [ ]:
price.dtypes

In [ ]:
plt.plot(price['시점'], price['매매가격지수'])
plt.title('Trend of 매매가격지수')
plt.xlabel('시점')
plt.ylabel('매매가격지수')
plt.xticks(rotation=45)
plt.xticks(price['시점'], rotation=45)
plt.show()
# 우상향하는 추세로 보임

### @title 1. 금리 (한국은행)

In [ ]:
service_key = "YOUR_API_KEY" # api 인증키
search_s = '20210101' # 기준금리 조회 시작 날짜
search_e = '20230331' # 기준금리 조회 종료 날짜

In [ ]:
!pip install PublicDataReader

In [ ]:
from PublicDataReader import Ecos

api = Ecos(service_key)

base_rate = api.get_statistic_search(통계표코드="722Y001", 주기="D", 검색시작일자=search_s, 검색종료일자=search_e)
base_rate = base_rate[base_rate['통계항목명1'] == '한국은행 기준금리'][['시점', '값']]

In [ ]:
base_rate['시점'] = pd.to_datetime(base_rate['시점'])
base_rate['값'] = base_rate['값'].astype(float)
base_rate.set_index('시점', inplace=True)
base_rate.info()

In [ ]:
base_rate.head

In [ ]:
base_rate.plot(title='Base Rate')

In [ ]:
base_rate

In [ ]:
# 금리의 경우 매일매일의 값이 존재하기 때문에 월 단위로 바꾸는 전처리

rate = base_rate.resample('MS').first()

rate.head()

In [ ]:
rate.index = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20',
                '21', '22', '23', '24', '25', '26']
rate['시점'] = ['2021-1', '2021-2', '2021-3', '2021-4', '2021-5', '2021-6', '2021-7', '2021-8', '2021-9', '2021-10', '2021-11', '2021-12',
                 '2022-1', '2022-2', '2022-3', '2022-4', '2022-5', '2022-6', '2022-7', '2022-8', '2022-9', '2022-10', '2022-11', '2022-12',
                 '2023-1', '2023-2', '2023-3']
rate.head()

In [ ]:
rate.columns = ['금리', '시점']
rate['시점'] = pd.to_datetime(rate['시점'], format='%Y-%m')
rate.head()

In [ ]:
rate.info()

### @title 환율

In [ ]:
import yfinance as yf
from datetime import datetime

enddate=datetime.now().strftime('%Y-%m-%d')
kospi=yf.download('^KS11', '2021-01-01', '2023-03-31', auto_adjust=True)
kospi

### @title 2. GDP

In [ ]:
gdp = pd.read_csv('../data/국내총생산에 대한 지출_1.csv')
gdp

In [ ]:
gdp = gdp.iloc[[20]]

In [ ]:
gdp.reset_index(drop=True, inplace=True)
gdp

In [ ]:
gdp1 = gdp.transpose()
gdp1 = gdp1.iloc[4:-2 , :2]

print(gdp1)

In [ ]:
gdp1.index = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16']
gdp1['시점'] = ['2019-1',	'2019-4', '2019-7','2019-10','2020-1','2020-4','2020-7','2020-10','2021-1', '2021-4',	'2021-7', '2021-10','2022-1','2022-4','2022-7','2022-10','2023-1']
gdp1

In [ ]:
gdp1.columns = ['국내총생산', '시점']
gdp1

In [ ]:
gdp1['시점'] = pd.to_datetime(gdp1['시점'], format='%Y-%m')

In [ ]:
### 분기별 데이터라서 직접 남은 달의 빈값 채워줘야 됨

gdp1.set_index('시점', inplace=True)

gdp1 = gdp1.resample('M').ffill()

gdp1.reset_index(inplace=True)

gdp1['시점'] = gdp1['시점'].dt.strftime('%Y-%m')

gdp1

In [ ]:
gdp2 = pd.DataFrame({'시점':['2023-2', '2023-3'], '국내총생산':['523,816.2', '523,816.2']})
gdp2

In [ ]:
gdp1 = pd.concat([gdp1, gdp2], ignore_index=True)

In [ ]:
gdp1['시점'] = pd.to_datetime(gdp1['시점'], format='%Y-%m')
gdp1.dtypes

In [ ]:
gdp1

In [ ]:
# ACF 검정 - 비정상적 시계열임을 알 수 있음

def acf_plot(data, N_LAGS, alpha):
    from statsmodels.graphics.tsaplots import plot_acf
    import matplotlib.pyplot as plt

    data = data.str.replace(',', '').astype(float)

    fig = plot_acf(data, lags=N_LAGS, alpha=alpha)
    plt.xlabel(f'Lag at 0 to {N_LAGS}')
    plt.ylabel("lag at k's autocorrelation")
    plt.show()

acf_plot(gdp1['국내총생산'], 10, 0.05)

In [ ]:
# PACF 검정 - 시차 1까지 비정상적 시계열임을 알 수 있음

def pacf_plot(data, N_LAGS, pval):
    from statsmodels.graphics.tsaplots import plot_pacf

    data = data.str.replace(',', '').astype(float)

    plot_pacf(data, lags=N_LAGS, alpha=pval, method='ywm')
    plt.xlabel(f'Lag at k (0 to {N_LAGS})')
    plt.ylabel("lag at k's Partial autocorrelation")
    plt.show()

pacf_plot(gdp1['국내총생산'], 10, 0.05)

### @title 3. 대출금리

In [ ]:
loan = pd.read_csv('../data/예금은행 대출금리.csv')
loan.head()

In [ ]:
loan1 = loan.transpose()

loan1 = loan1.iloc[5:, :2]
loan1

In [ ]:
loan1.index = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20',
                '21', '22', '23', '24', '25', '26']
loan1['시점'] = ['2023-3',	'2023-2',	'2023-1','2022-12','2022-11','2022-10','2022-9','2022-8','2022-7','2022-6', '2022-5', '2022-4',
                '2022-3',	'2022-2',	'2022-1', '2021-12','2021-11','2021-10','2021-9','2021-8','2021-7','2021-6','2021-5', '2021-4',
                 '2021-3',	'2021-2',	'2021-1']
loan1.head()

In [ ]:
loan1.columns = ['대출금리', '시점']
loan1['시점'] = pd.to_datetime(loan1['시점'], format='%Y-%m')
loan1.head()

In [ ]:
## 2023년부터이므로 역순으로 재구성해야 됨

loan2 = loan1[::-1].reset_index(drop=True)
loan2.head()

In [ ]:
loan2.dtypes

In [ ]:
# ACF 검정 - 비정상적 시계열임을 알 수 있음

def acf_plot(data, N_LAGS, alpha):
    from statsmodels.graphics.tsaplots import plot_acf
    import matplotlib.pyplot as plt

    fig = plot_acf(data, lags=N_LAGS, alpha=alpha)
    plt.xlabel(f'Lag at 0 to {N_LAGS}')
    plt.ylabel("lag at k's autocorrelation")
    plt.show()

acf_plot(loan2['대출금리'], 10, 0.05)

In [ ]:
# PACF 검정 - 시차 1까지 비정상적 시계열임을 알 수 있음

def pacf_plot(data, N_LAGS, pval):
    from statsmodels.graphics.tsaplots import plot_pacf

    plot_pacf(data, lags=N_LAGS, alpha=pval, method='ywm')
    plt.xlabel(f'Lag at k (0 to {N_LAGS})')
    plt.ylabel("lag at k's Partial autocorrelation")
    plt.show()

pacf_plot(loan2['대출금리'], 10, 0.05)

### @title 4. (아파트) 매매 거래량

In [ ]:
pro = pd.read_excel('../data/매매 거래량.xlsx')
pro.head()

In [ ]:
pro1 = pro.transpose()

pro1 = pro1.iloc[3:-1, :2]
pro1

In [ ]:
pro1.index = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20',
                '21', '22', '23', '24', '25', '26']
pro1['시점'] = ['2021-1', '2021-2', '2021-3', '2021-4', '2021-5', '2021-6', '2021-7', '2021-8', '2021-9', '2021-10', '2021-11', '2021-12',
                 '2022-1', '2022-2', '2022-3', '2022-4', '2022-5', '2022-6', '2022-7', '2022-8', '2022-9', '2022-10', '2022-11', '2022-12',
                 '2023-1', '2023-2', '2023-3']
pro1.head()

In [ ]:
pro1.columns = ['매매 거래량', '시점']
pro1['시점'] = pd.to_datetime(pro1['시점'], format='%Y-%m')
pro1.head()

In [ ]:
# ACF 검정 - 비정상적 시계열임을 알 수 있음

def acf_plot(data, N_LAGS, alpha):
    from statsmodels.graphics.tsaplots import plot_acf

    data = data.str.replace(',', '').astype(float)

    fig = plot_acf(data, lags=N_LAGS, alpha=alpha)
    plt.xlabel(f'Lag at 0 to {N_LAGS}')
    plt.ylabel("lag at k's autocorrelation")
    plt.show()

acf_plot(pro1['매매 거래량'], 10, 0.05)

In [ ]:
# PACF 검정 - 시차 1까지 비정상적 시계열임을 알 수 있음


def pacf_plot(data, N_LAGS, pval):
    from statsmodels.graphics.tsaplots import plot_pacf
    data = data.str.replace(',', '').astype(float)

    plot_pacf(data, lags=N_LAGS, alpha=pval, method='ywm')
    plt.xlabel(f'Lag at k (0 to {N_LAGS})')
    plt.ylabel("lag at k's Partial autocorrelation")
    plt.show()

pacf_plot(pro1['매매 거래량'], 10, 0.05)

### @title 네이버 데이터랩 검색량

In [ ]:
apa_df = pd.read_excel('../data/아파트 (네이버).xlsx')
apa_df.head()

In [ ]:
# 네이버 데이터랩 '아파트' 키워드량 - 상대적인 수치이므로 백분율로 변환하는 전처리

apa_df['아파트'] = apa_df['아파트'].apply(lambda x: round(float(x), 3) * 0.01)
apa_df.head()

### @title 데이터프레임 merge

In [ ]:
df = pd.merge(price, rate, on='시점')
df = pd.merge(df, gdp1, on='시점')
df = pd.merge(df, loan2, on='시점')
df = pd.merge(df, pro1, on='시점')
df.head()

In [ ]:
df

In [ ]:
df.dtypes

In [ ]:
gdp1.head()

In [ ]:
df = pd.read_excel('../data/시계열1.xlsx')
df.head()

In [ ]:
df = df.drop(labels='국내총생산', axis=1)
df.head()

In [ ]:
df = pd.merge(df, gdp1, on='시점')
df.head()

In [ ]:
df = df[['시점', '매매가격지수', '금리', '국내총생산', '대출금리', '아파트거래량', '매매 거래량']]
df.head()

In [ ]:
df.to_excel('시계열_base.xlsx', index=False)

### @title 상관관계 분석

In [ ]:
# 1. 각 시계열의 시차를 고려하지 않고 상관관계 분석

df['매매가격지수'] = pd.to_numeric(df['매매가격지수'], errors='coerce')
df['국내총생산'] = pd.to_numeric(df['국내총생산'], errors='coerce')
df['대출금리'] = pd.to_numeric(df['대출금리'], errors='coerce')
df['매매 거래량'] = pd.to_numeric(df['매매 거래량'], errors='coerce')
df.dtypes

In [ ]:
correlation = df.corr()
correlation

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', square=True)

# Customize the plot
plt.title('Correlation Heatmap')
plt.xticks(rotation=45)
plt.yticks(rotation=0)

# Display the heatmap
plt.show()

In [ ]:
# 교차상관관계 살펴보기 (두 변수씩 보면 됨)
# 정상성을 만족하도록 만들어야 됨 -> ADF 검정, ACF, PACF 통해 정상성 확인 후 차분여부 결정
# price - 매매가격지수, rate- 금리, gdp1 - 국내총생산, loan1 - 대출금리, pro1 - 매매 거래량